# Modelo preditivo v0

Prevê o pico de ameaça do ciclo (`max_threat_score`) a partir das features defensivas agregadas, só para ciclos que começam no goleiro com 11 defensores atrás da bola. Versão naive: sem seleção fina de features, sem tuning.

## 1. Setup

Imports, sessão Spark, caminhos e funções auxiliares (`evaluate`: MAE, RMSE, MAPE e R²; `run_cv`: `cross_validate` com média ± desvio). MAPE é uma fração (0.25 = 25%) e pode explodir quando `y` é próximo de zero.

In [1]:
import json
import pandas as pd
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

sys.path.append(str(Path().resolve().parent.parent))  # raiz do projeto, pra importar src.utils
from src.utils.spark_aggregation import build_feature_target_stats_df

pd.set_option('display.max_columns', None)

In [2]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .master("local[*]")
    .appName("performance_prediction")
    .getOrCreate()
)

In [3]:
data_folder_path = Path().resolve().parent.parent / "data"

In [4]:
SCORING = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAPE": "neg_mean_absolute_percentage_error",
    "R2": "r2",
}


def evaluate(name, y_true, y_pred):
    """MAE, RMSE, MAPE e R² — imprime e devolve como dict."""
    metrics = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,  # raiz manual do MSE
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }
    print(f"{name} — " + " | ".join(f"{k}: {v:.4f}" for k, v in metrics.items()))
    return metrics


def run_cv(name, model, X, y, cv):
    """cross_validate; imprime média ± desvio e devolve as médias."""
    results = cross_validate(model, X, y, cv=cv, scoring=SCORING)
    means = {}
    for metric in SCORING:
        scores = results[f"test_{metric}"]
        if metric != "R2":
            scores = -scores  # métricas "neg_*" vêm negativas por convenção do sklearn
        means[metric] = scores.mean()
        print(f"{name} (CV) — {metric}: {scores.mean():.4f} ± {scores.std():.4f}")
    return means

## 2. Dados e pré-processamento

Leitura, filtro dos ciclos, features relevantes, agregação por ciclo e split treino/teste.

### 2.1 Dados

Lê a base de ameaça (parquet) e a base de features (csv).

In [5]:
threat_dataset_path = str(data_folder_path / "threat_dataset")
features_dataset_path = str(data_folder_path / "features_dataset")

df_threat = spark.read.parquet(threat_dataset_path)
df_features = spark.read.csv(features_dataset_path, header=True, inferSchema=True)

### 2.2 Ciclos de posse

Junta ameaça + features por evento, remove eventos com tracking inconsistente (`is_tracking_inconsistent`) e mantém só os ciclos que **começam no goleiro com 11 defensores atrás da bola** (mesmo critério de `eda_cycle_start_gk.ipynb`).

In [6]:
# só as colunas necessárias pro filtro de ciclos, pra agregação e pro target
threat_cols = [
    'gameId',
    'competitionId',
    'season',
    'eventId',
    'startGameClock',
    'possession_id',
    'eventPlayerPositionType',
    'defenders',
    'threat_score',
]

In [7]:
df_threat_features = (
    df_threat.select(*threat_cols)
    .join(df_features, on=["competitionId", "season", "gameId", "eventId"], how="inner")
    .filter(~F.col("is_tracking_inconsistent"))  # remove eventos com tracking inconsistente
)

# flag: evento é o primeiro da sua posse
w_pos = Window.partitionBy("competitionId", "season", "gameId").orderBy("startGameClock")

df_threat_features = df_threat_features.withColumn(
    "first_cycle_event",
    F.lag("possession_id", 1).over(w_pos).isNull() |
    (F.lag("possession_id", 1).over(w_pos) != F.col("possession_id"))
)

In [8]:
# ciclos cujo 1º evento é do goleiro com 11 defensores; o join traz todos os eventos desses ciclos
df_cycle_start_gk_keys = (
    df_threat_features
    .filter(
        F.col("first_cycle_event") &
        (F.col("eventPlayerPositionType") == "GK") &
        (F.col("defenders") == 11)
    )
    .select("competitionId", "season", "gameId", "possession_id")
)

df_cycle_start_gk = (
    df_threat_features
    .join(F.broadcast(df_cycle_start_gk_keys), on=["competitionId", "season", "gameId", "possession_id"], how="inner")
    .cache()
)
df_cycle_start_gk.show(5)

+-------------+---------+------+-------------+--------------------+--------------+-----------------------+---------+------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+
|competitionId|   season|gameId|possession_id|             eventId|startGameClock|eventPlayerPositionType|defenders|threat_score|is_tracking_inconsistent|surface_area|stretch_index|team_length|team_width|defense_width|height_goal_player|height_goal_team_centroide|height_goal_def_centroide|def_mid_dist|def_atk_dist|atk_mid_dist|numeric_superiority_10m|numeric_superiority_20m|first_cycle_event|
+-------------+---------+------+-------------+--------------------+--------------+-----------------------+---------+------------+------------------------+------------+-------------+-----------+----------+----

### 2.3 Features relevantes

Lê `data/relevant_features.json` (gerado em `eda_cycle_start_gk.ipynb`): para cada feature, as agregações mais correlacionadas com o pico de ameaça (com p-valor < 0.05). **Só essas colunas entram no modelo** — features sem nenhuma agregação significativa ficam de fora.

In [9]:
with open(data_folder_path / "relevant_features.json", encoding="utf-8") as f:
    relevant_features = json.load(f)

# features descartadas do modelo
EXCLUDED_FEATURES = ["height_goal_player", "height_goal_def_centroide", "height_goal_team_centroide","numeric_superiority_10m" ,"numeric_superiority_20m"]
relevant_features = {feature: aggs for feature, aggs in relevant_features.items() if feature not in EXCLUDED_FEATURES}

# features com ao menos uma agregação relevante
features = [feature for feature, aggs in relevant_features.items() if aggs]

# colunas do X: "{agregação}_{feature}" para cada agregação do json
feature_cols = [
    f"{next(key for key in agg if key != 'p-value')}_{feature}"
    for feature, aggs in relevant_features.items()
    for agg in aggs
]

print(f"{len(features)} features / {len(feature_cols)} colunas no X")
print(feature_cols)

8 features / 16 colunas no X
['min_surface_area', 'max_surface_area', 'min_stretch_index', 'max_stretch_index', 'min_team_length', 'median_team_length', 'max_team_width', 'min_team_width', 'max_defense_width', 'min_defense_width', 'min_def_mid_dist', 'max_def_mid_dist', 'min_def_atk_dist', 'max_def_atk_dist', 'min_atk_mid_dist', 'max_atk_mid_dist']


### 2.4 Agregação por ciclo

Uma linha por ciclo: agrega as features relevantes e o target com `build_feature_target_stats_df` (`src/utils/spark_aggregation.py`).

In [10]:
target = "threat_score"
group_cols = ["competitionId", "season", "gameId", "possession_id"]

df_features_target_agg = build_feature_target_stats_df(
    df=df_cycle_start_gk,
    features=features,
    target=target,
    group_cols=group_cols,
)

df_model_pd = df_features_target_agg.toPandas()
df_model_pd.shape

(5570, 47)

### 2.5 Dataset do modelo

Monta `X` (colunas do json) e `y` (`max_threat_score`), remove ciclos sem target, separa treino/teste e preenche os NaN das features pela mediana do treino. O teste fica guardado (hold-out) até a avaliação final.

In [11]:
df_model_pd.head()

,competitionId,season,gameId,possession_id,avg_surface_area,median_surface_area,std_surface_area,min_surface_area,max_surface_area,avg_stretch_index,median_stretch_index,std_stretch_index,min_stretch_index,max_stretch_index,avg_team_length,median_team_length,std_team_length,min_team_length,max_team_length,avg_team_width,median_team_width,std_team_width,min_team_width,max_team_width,avg_defense_width,median_defense_width,std_defense_width,min_defense_width,max_defense_width,avg_def_mid_dist,median_def_mid_dist,std_def_mid_dist,min_def_mid_dist,max_def_mid_dist,avg_def_atk_dist,median_def_atk_dist,std_def_atk_dist,min_def_atk_dist,max_def_atk_dist,avg_atk_mid_dist,median_atk_mid_dist,std_atk_mid_dist,min_atk_mid_dist,max_atk_mid_dist,avg_threat_score,sum_threat_score,max_threat_score
0,1,2022-2023,4727,28,784.475,763.820,167.812,602.18,1008.08,14.820,14.600,1.261,13.56,16.52,36.450,35.495,6.750,29.25,45.56,38.213,38.285,0.975,36.95,39.33,38.213,38.285,0.975,36.95,39.33,12.945,12.250,2.349,11.02,16.26,19.063,17.970,3.975,15.87,24.44,6.120,6.000,1.717,4.29,8.19,0.303,1.213,0.399
1,1,2022-2023,4727,73,1069.770,1069.770,NaN,1069.77,1069.77,16.920,16.920,NaN,16.92,16.92,34.090,34.090,NaN,34.09,34.09,37.180,37.180,NaN,37.18,37.18,32.920,32.920,NaN,32.92,32.92,13.750,13.750,NaN,13.75,13.75,32.080,32.080,NaN,32.08,32.08,18.330,18.330,NaN,18.33,18.33,0.291,0.291,0.291
2,1,2022-2023,4727,137,858.504,897.425,186.415,505.25,1074.88,15.623,16.090,1.744,12.26,17.97,31.749,31.745,4.900,23.30,38.72,39.868,39.515,3.540,34.07,47.20,35.137,35.720,1.675,32.82,37.57,12.637,13.045,3.701,7.59,16.91,26.808,27.545,5.478,17.98,34.45,14.172,14.175,2.733,8.93,17.54,0.349,3.490,0.575
3,1,2022-2023,4727,177,1081.108,1191.070,339.381,605.60,1336.69,17.735,18.835,3.487,12.93,20.34,44.360,46.655,6.894,34.30,49.83,34.070,34.025,6.834,26.12,42.11,28.935,31.275,5.689,20.47,32.72,16.097,18.035,4.734,9.07,19.25,36.387,38.385,7.509,26.09,42.69,20.293,20.280,3.722,17.02,23.59,0.383,1.533,0.541
4,1,2022-2023,4727,193,693.587,669.410,135.571,469.10,967.39,14.174,14.270,1.584,11.61,17.07,29.916,29.870,3.474,18.38,36.93,36.867,36.430,4.060,26.57,44.01,35.446,34.900,4.630,24.92,43.55,9.788,8.220,3.516,5.77,16.65,18.648,18.510,2.813,12.05,23.95,8.860,9.620,3.431,0.81,16.10,0.469,9.845,0.763


In [12]:
target_col = f"max_{target}"

# sklearn não aceita NaN (ex: std de ciclo com 1 evento) — descarta, sem imputação na v0
#df_model_pd = df_model_pd.dropna(subset=feature_cols + [target_col])

X = df_model_pd[feature_cols]
y = df_model_pd[target_col]

print(f"{len(df_model_pd)} ciclos | {X.shape[1]} features")

5570 ciclos | 16 features


In [13]:
# split antes de qualquer treino: o teste só é usado no passo 7
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# NaN das features -> mediana do treino (aplicada também no teste, sem vazamento)
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

Treino: 4456 | Teste: 1114


## 3. Modelagem

Baseline, cross-validation, comparação e avaliação final no hold-out.

### 3.1 Baseline trivial

Prevê sempre a média do treino, ignorando as features. É o piso que os modelos precisam superar.

In [14]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

baseline_metrics = evaluate("Baseline (teste)", y_test, baseline_model.predict(X_test))

Baseline (teste) — MAE: 0.1454 | RMSE: 0.1683 | MAPE: 0.3549 | R2: -0.0093


### 3.2 Cross-validation

KFold de 5 folds com seed fixa. Os mesmos folds valem pra todos os modelos (comparação justa) e rodam só dentro do treino.

In [15]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

### 3.3 Regressão linear (CV)

Avalia a regressão linear nos 5 folds do treino.

In [16]:
linear_metrics = run_cv("Regressão Linear", LinearRegression(), X_train, y_train, kfold)

Regressão Linear (CV) — MAE: 0.0744 ± 0.0018
Regressão Linear (CV) — RMSE: 0.1013 ± 0.0019
Regressão Linear (CV) — MAPE: 0.1761 ± 0.0063
Regressão Linear (CV) — R2: 0.5943 ± 0.0140


### 3.4 Árvore de decisão (CV)

Mesmo procedimento da regressão linear. Depois, treina a árvore uma vez no treino inteiro só para listar as `feature_importances_`.

In [17]:
tree_metrics = run_cv("Árvore de Decisão", DecisionTreeRegressor(random_state=42), X_train, y_train, kfold)

Árvore de Decisão (CV) — MAE: 0.0911 ± 0.0033
Árvore de Decisão (CV) — RMSE: 0.1368 ± 0.0040
Árvore de Decisão (CV) — MAPE: 0.2101 ± 0.0079
Árvore de Decisão (CV) — R2: 0.2598 ± 0.0343


In [18]:
# cross_validate não devolve o modelo treinado; este fit é só pra inspecionar as importâncias
tree_inspect = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

pd.Series(tree_inspect.feature_importances_, index=feature_cols).sort_values(ascending=False)

min_team_length       0.198620
min_defense_width     0.153206
max_stretch_index     0.132643
min_def_atk_dist      0.069452
min_team_width        0.056073
max_def_atk_dist      0.051509
min_stretch_index     0.047410
min_def_mid_dist      0.040175
max_surface_area      0.039901
min_surface_area      0.036570
max_atk_mid_dist      0.035854
median_team_length    0.035376
max_defense_width     0.027599
max_team_width        0.025797
max_def_mid_dist      0.025340
min_atk_mid_dist      0.024476
dtype: float64

### 3.5 Comparação

Baseline: métricas do teste. Regressão linear e árvore: média do CV.

In [19]:
comparison_df = pd.DataFrame(
    {"Baseline (média)": baseline_metrics, "Regressão Linear": linear_metrics, "Árvore de Decisão": tree_metrics}
).T.round(4)

comparison_df

,MAE,RMSE,MAPE,R2
Baseline (média),0.1454,0.1683,0.3549,-0.0093
Regressão Linear,0.0744,0.1013,0.1761,0.5943
Árvore de Decisão,0.0911,0.1368,0.2101,0.2598


### 3.6 Modelo final

Escolhe entre regressão linear e árvore pelo maior R² médio do CV e retreina no treino inteiro (sem folds).

In [20]:
candidates = {
    "Regressão Linear": (LinearRegression(), linear_metrics),
    "Árvore de Decisão": (DecisionTreeRegressor(random_state=42), tree_metrics),
}

final_model_name = max(candidates, key=lambda name: candidates[name][1]["R2"])
final_model = candidates[final_model_name][0].fit(X_train, y_train)

print(f"Modelo escolhido: {final_model_name}")

Modelo escolhido: Regressão Linear


### 3.7 Avaliação no hold-out

Única vez que o teste é usado. O CV serve para comparar/escolher modelos; o hold-out mede como o modelo escolhido generaliza para dados nunca vistos. **Resultado final da v0.**

In [21]:
final_metrics = evaluate(f"{final_model_name} (hold-out)", y_test, final_model.predict(X_test))

Regressão Linear (hold-out) — MAE: 0.0754 | RMSE: 0.1014 | MAPE: 0.1733 | R2: 0.6334
